# Store files in R2 and continue the learning experiment

If your files are already in R2, start with **COMPARE on CPU**. Section 6 compares the earlier Drive run with the later R2 run using their saved reports. It does no training or dataset download.

Then open a fresh **A100 session with BATCH**. Section 7 reuses the completed beta-0.01 result and trains beta 0, 0.001, 0.1, and 1 from the same original model before task 17. Results are saved to R2 and the GPU is released automatically at the end.

COPY is for the initial file migration. RUN retains the original single-lesson workflow.

The beta batch is now complete. For the next check, choose a fresh A100 session, set MODE = "REPEAT" in section 2, and run all cells from the top. Section 8 repeats only beta 0.1 from the original starting model and compares it with the saved beta-0.1 trial.

Before starting, connect your storage:

1. In Cloudflare, open **R2 object storage**. If it is not enabled, review the billing terms before enabling it.
2. Create a private bucket named `uncle-experiments` using Standard storage. A bucket is a named place for files. If you use another name, change BUCKET below.
3. Open **Manage API Tokens** on the R2 overview. Create an R2 token with **Object Read & Write**, limited to this bucket. Save its **Access Key ID** and **Secret Access Key**.
4. Find the bucket's **S3 API endpoint**. Use the HTTPS server address without a trailing slash or bucket name. Keep any jurisdiction shown in the address.
5. Click the key icon in Colab's left sidebar. Add `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` to **Secrets**, then turn on notebook access for each. Do not paste credentials into cells, Git, or chat.

R2 charges for storage and some requests. Check [Cloudflare's pricing](https://developers.cloudflare.com/r2/pricing/) before copying a large collection. File verification reads every uploaded file back.

COPY transfers experiment files under DRIVE_ROOT. It skips folders named `data`, `datasets`, or `tiny-imagenet-200`, along with JPEG images and archives. Code stays in Git. Training downloads Tiny ImageNet once per fresh session and reuses it for every beta trial. Use one session to write to each experiment folder.

## 1. Load the code

Run this cell in each fresh session. It loads a fixed code version and the file-transfer library.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
REPO = Path("/content/uncle-r2-learning")
CODE_VERSION = "dd7beb173fb957467116c71b6ffe39e562799016"
assert not any(n == "uncle" or n.startswith("uncle.") for n in sys.modules), "Start a fresh runtime before loading this code again."
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), "Use a fresh session."
    remote = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip().removesuffix(".git")
    assert remote == "https://github.com/sumitasthana/CARK", "This folder belongs to another project."
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Save local code changes and use a fresh session."
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "boto3>=1.36"], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Code version:", CODE_VERSION)

## 2. Connect to your bucket

For the next check, choose REPEAT in a fresh A100 session. COMPARE and BATCH remain available for the earlier steps. Use COPY only if experiment files still need migration, or RUN for a separate single-lesson run. If your bucket has another name, change BUCKET. The source file below is the model you already confirmed; there is no file number to choose.

EXPERIMENT_ID identifies the completed R2 reference run. Keep it unchanged for COMPARE and BATCH. BATCH_EXPERIMENT_ID identifies the four new trials; use a new batch name after an interrupted batch. The two report keys and comparison result key are set below.

In [ ]:
import json
import math
import torch
from google.colab import userdata
from uncle.storage import R2Store
from uncle.learning_diagnostic import inspect_source, file_hash, run_learning_diagnostic, summarize_diagnostic

MODE = "COMPARE"  # CPU: COMPARE first. Fresh A100: BATCH afterwards.
BUCKET = "uncle-experiments"
EXPERIMENT_ID = "20261006_learning_loss_r2_01"
BETAS = (0.01,)  # Used only by the original RUN mode.
BATCH_EXPERIMENT_ID = "20261007_beta_batch_01"
BATCH_BETAS = (0.0, 0.001, 0.01, 0.1, 1.0)
REPEAT_EXPERIMENT_ID = "20261007_beta01_repeat_01"
REPEAT_REPORT_KEY = "uncle/learning_loss_diagnostic/20261007_beta_batch_01/beta_0_1/report.json"
DRIVE_REPORT_KEY = "uncle/learning_loss_diagnostic/20261006_learning_loss_01/beta_0_01/report.json"
R2_REPORT_KEY = "uncle/learning_loss_diagnostic/20261006_learning_loss_r2_01/beta_0_01/report.json"
COMPARISON_KEY = "uncle/learning_loss_diagnostic/report_comparison/20261007_drive_vs_r2_01.json"
SOURCE_KEY = "uncle/learning_initialization/20261006_hyperfan_L3_L0_L9_seed0_02/checkpoint_seq1_resnet50_seed0.pt"
DRIVE_ROOT = Path("/content/drive/MyDrive/uncle")
COPY_PREFIX = "uncle"
EXCLUDE_DIRS = ("data", "datasets", "tiny-imagenet-200", ".git", "__pycache__", ".ipynb_checkpoints")
EXPERIMENT_SUFFIXES = (".pt", ".pth", ".ckpt", ".json", ".csv", ".txt", ".log", ".yaml", ".yml", ".md", ".png", ".svg", ".pdf", ".npy", ".npz")
CACHE = Path("/content/r2-cache")
OUTPUT_ROOT = Path("/content/r2-results") / EXPERIMENT_ID
OUTPUT_PREFIX = "uncle/learning_loss_diagnostic/" + EXPERIMENT_ID
DATA = Path("/content/data/tiny-imagenet-200")
NEW_TASK = "17"
assert MODE in ("COPY", "RUN", "COMPARE", "BATCH", "REPEAT")
assert EXPERIMENT_ID and all(c.isalnum() or c in "_-" for c in EXPERIMENT_ID)
assert BETAS and len(set(BETAS)) == len(BETAS)
assert all(math.isfinite(b) and b >= 0 for b in BETAS)
store = R2Store.connect(userdata.get("R2_ENDPOINT"), BUCKET,
    userdata.get("R2_ACCESS_KEY_ID"), userdata.get("R2_SECRET_ACCESS_KEY"))
# Listing one page checks that the bucket exists and the credentials work.
next(iter(store.keys(COPY_PREFIX + "/")), None)
print("Connected to R2 bucket:", BUCKET)
print("Mode:", MODE)

## 3. Copy Drive files once, then check the source model

In COPY mode, allow Drive access. The notebook copies model files, reports, tables, text logs, and figures, then reads them back to check their contents. Dataset folders, JPEG images, and archives are skipped. Originals stay on Drive. If the copy stops, run COPY again; matching experiment files are checked instead of uploaded again. Close other notebooks that are saving to this folder first.

In RUN mode, this cell downloads only the chosen model. It checks the tasks and scores saved inside it. It stops if this is a different model.

In [ ]:
if MODE == "COPY":
    assert not torch.cuda.is_available(), "Use a CPU session for the one-time copy."
    from google.colab import drive
    drive.mount("/content/drive")
    def copied(row):
        print("Copied and checked:", row["key"], flush=True)
    print("Copying experiment files only. Skipping datasets, JPEG images, and archives.", flush=True)
    copied_files = store.copy_tree(DRIVE_ROOT, COPY_PREFIX, on_file=copied, exclude_dirs=EXCLUDE_DIRS, suffixes=EXPERIMENT_SUFFIXES)
    print("Copy finished:", copied_files["files"], "files checked.")
    print("Skipped folders:", copied_files["skipped_directories"])
elif MODE in ("RUN", "BATCH", "REPEAT"):
    assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Choose a fresh A100 runtime."
    if MODE == "RUN":
        assert not list(store.keys(OUTPUT_PREFIX + "/")), "This experiment has saved files. Read them first; use a new experiment name for a new run."
if MODE != "COMPARE":
    SOURCE = store.download(SOURCE_KEY, CACHE)
    details = inspect_source(SOURCE)
    assert details["completed_tasks"] == ["3", "0", "9", "5"], "Choose the confirmed model after task 5."
    assert details["config"]["initialization"] == "hyperfan_in"
    assert details["config"]["dataset"] == "tiny_imagenet" and details["config"]["backbone"] == "resnet50"
    assert details["accuracies"] == {"3": 27.0, "0": 31.2, "9": 48.4, "5": 54.8}, "Saved scores differ from your confirmed model. Stop and check the source."
    SOURCE_SHA256 = file_hash(SOURCE)
    print("Saved tasks:", details["completed_tasks"])
    print("Saved validation scores:", details["accuracies"])
    print("Model fingerprint:", SOURCE_SHA256)
    print("Next lesson: task", NEW_TASK, "only.")
else:
    print("COMPARE mode: no model or dataset download. The report check is in section 6.")


## 4. Teach task 17 and save results automatically

COPY mode skips this cell. RUN mode checks the starting scores on validation images, then teaches task 17. After every pass through the training images, it prints losses and scores and uploads the report. At the end it uploads and checks the final model, then saves the completed report.

Every beta loads the same source model independently. If Colab stops during training, the uploaded reports remain available. A final model is saved only when the lesson finishes. This notebook cannot continue midway through an interrupted lesson.

In [ ]:
reports = []
if MODE == "RUN":
    from uncle.config import Config
    from uncle.streams import build_tasks
    assert file_hash(SOURCE) == SOURCE_SHA256
    source_config = Config(**details["config"])
    task_order = (*details["completed_tasks"], NEW_TASK)
    tasks = build_tasks(source_config, root=DATA, include=task_order, download=True)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    selection = {"source_key": SOURCE_KEY, "source_sha256": SOURCE_SHA256,
        "completed_tasks": details["completed_tasks"], "new_task": NEW_TASK,
        "saved_accuracies": details["accuracies"], "code_version": CODE_VERSION, "betas": BETAS}
    (OUTPUT_ROOT / "selection.json").write_text(json.dumps(selection, indent=2) + "\n")
    store.upload(OUTPUT_ROOT / "selection.json", OUTPUT_PREFIX + "/selection.json")
    for beta in BETAS:
        label = format(beta, ".8g").replace(".", "_")
        output = OUTPUT_ROOT / f"beta_{label}"
        prefix = OUTPUT_PREFIX + f"/beta_{label}"
        def save_epoch(row):
            store.publish_lesson(output, prefix)
            print(f"Pass {row['epoch']}: new-task loss = {row['task_loss']:.4f}; beta times protection = {row['weighted_protection_loss']:.4f}")
            print("Validation scores:", " | ".join(f"task {t}: {row['accuracies'][t]:.1f}%" for t in task_order), flush=True)
            print("Report checked in R2.", flush=True)
        print("Loading the source model. Beta:", beta, flush=True)
        report = run_learning_diagnostic(SOURCE, output, NEW_TASK, tasks,
            beta=beta, device="cuda", code_version=CODE_VERSION, on_epoch=save_epoch)
        assert report["source_sha256"] == SOURCE_SHA256
        store.publish_lesson(output, prefix, complete=True)
        reports.append((output, prefix, report))
        print("Final model and report checked in R2:", prefix, flush=True)
        torch.cuda.empty_cache()
    comparisons = [summarize_diagnostic(report) for _, _, report in reports]
    (OUTPUT_ROOT / "beta_comparison.json").write_text(json.dumps(comparisons, indent=2) + "\n")
    store.upload(OUTPUT_ROOT / "beta_comparison.json", OUTPUT_PREFIX + "/beta_comparison.json")
    for row in comparisons:
        print(f"Beta {row['beta']:g}: task 17 = {row['new_task_accuracy']:.1f}%; mean old score change = {row['mean_old_change']:+.1f} points")
    print("Training and uploads finished. Run the final cell to check history and release the GPU.")
else:
    print("This original training cell is skipped in", MODE, "mode.")

## 5. Check saved history and close the session

RUN mode checks that the final model adds exactly one lesson and preserves the earlier history. The model and report have already been checked in R2. After these checks, the GPU is released.

COPY mode disconnects Drive. COMPARE and BATCH skip this original closing cell; continue to sections 6 and 7. Their cells explain when the session is finished.

In [ ]:
if MODE in ("COPY", "RUN"):
    from google.colab import runtime
    if MODE == "RUN":
        from uncle import checkpoint as checkpointing
        assert reports and len(reports) == len(BETAS)
        for output, prefix, report in reports:
            saved_report = json.loads((output / "report.json").read_text())
            assert saved_report == report and report["status"] == "complete"
            saved_config = dict(report["config"])
            saved_config["tasks"] = tuple(saved_config["tasks"])
            saved_config["hidden"] = tuple(saved_config["hidden"])
            saved_config["requests"] = tuple(tuple(r) for r in saved_config["requests"])
            final = checkpointing.load(output / "checkpoint.pt", Config(**saved_config))
            assert len(final["history"]) == len(report["source_history"]) + 1
            assert json.loads(json.dumps(final["history"][:-1])) == report["source_history"]
            assert final["history"][-1]["task"] == NEW_TASK
            assert final["previous"] == report["final_accuracies"]
            assert len(report["epochs"]) == report["config"]["epochs"]
            assert not final["forgotten"]
            remote_report = store.download(prefix + "/report.json", CACHE)
            assert json.loads(remote_report.read_text()) == report
            remote_model = store.info(prefix + "/checkpoint.pt")
            assert remote_model["Metadata"]["sha256"] == file_hash(output / "checkpoint.pt")
            print("Checked history and R2 files:", prefix)
            del final
        print("Files checked in R2. Ending the GPU session.", flush=True)
    else:
        assert copied_files["files"] > 0
        print("Drive files copied and checked. Next: fresh A100 session, MODE = RUN.", flush=True)
        drive.flush_and_unmount(timeout_ms=600000)
    if MODE == "RUN":
        runtime.unassign()
else:
    print("Continue to section 6 for COMPARE, section 7 for BATCH, or section 8 for REPEAT.")


## 6. Compare the two saved reports on CPU

Choose a CPU runtime and MODE = "COMPARE", then run from the top. This cell downloads two small report files from R2. The earlier Drive report must have been included in COPY; Drive does not need to be mounted here.

It checks the starting-model fingerprint, starting scores, training settings, task sizes, and task being learned. It also prints software and GPU differences and final scores. A different code-version label is expected because storage support was added. Matching settings do not prove repeatability or explain the accuracy difference.

The comparison is saved to R2. When it says the starting model and training settings match, the next step is a fresh A100 session with MODE = "BATCH". If they do not match, check the printed differences before spending more GPU time. Do not repeat training to perform this check.

In [ ]:
if MODE == "COMPARE":
    assert not torch.cuda.is_available(), "Use a CPU runtime for the report comparison."
    report_keys = {"Earlier Drive run": DRIVE_REPORT_KEY, "Later R2 run": R2_REPORT_KEY}
    saved = {name: json.loads(store.download(key, CACHE).read_text()) for name, key in report_keys.items()}
    earlier, later = saved.values()
    assert all(r["status"] == "complete" for r in saved.values()), "Both reports must be completed runs."
    def differences(left, right):
        return {key: {"earlier": left.get(key), "later": right.get(key)}
                for key in sorted(set(left) | set(right)) if left.get(key) != right.get(key)}
    # Device choice is printed separately; the remaining config defines this lesson.
    training_settings = [{k: v for k, v in r["config"].items() if k != "device"} for r in saved.values()]
    settings_differences = differences(*training_settings)
    runtime_keys = ("python", "torch", "cuda", "gpu", "gpu_total_bytes", "device", "git_commit")
    runtimes = [{k: r["environment"].get(k) for k in runtime_keys} for r in saved.values()]
    checks = {
        "same_starting_model": earlier["source_sha256"] == later["source_sha256"],
        "same_starting_scores": earlier["starting_accuracies"] == later["starting_accuracies"],
        "same_training_settings": not settings_differences,
        "same_lesson": earlier["new_task"] == later["new_task"] and earlier["protected_tasks"] == later["protected_tasks"],
        "same_task_sizes": earlier["environment"].get("task_sizes") is not None and earlier["environment"].get("task_sizes") == later["environment"].get("task_sizes"),
    }
    comparison = {"report_keys": report_keys, "checks": checks,
        "ready_for_beta_batch": all(checks.values()),
        "source_hashes": {name: r["source_sha256"] for name, r in saved.items()},
        "report_hashes": {name: file_hash(store.download(key, CACHE)) for name, key in report_keys.items()},
        "starting_accuracies": {name: r["starting_accuracies"] for name, r in saved.items()},
        "settings_differences": settings_differences,
        "runtime_differences": differences(*runtimes),
        "code_versions": {name: r["code_version"] for name, r in saved.items()},
        "final_accuracies": {name: r["final_accuracies"] for name, r in saved.items()}}
    for name, matched in checks.items():
        print(name.replace("_", " ") + ":", "MATCH" if matched else "DIFFERENT")
    print("Training-setting differences:", json.dumps(settings_differences, indent=2))
    print("Computer and software differences:", json.dumps(comparison["runtime_differences"], indent=2))
    print("Code versions:", comparison["code_versions"])
    print("Task | earlier final score | later final score | later minus earlier")
    for task in (*later["protected_tasks"], later["new_task"]):
        a, b = earlier["final_accuracies"][task], later["final_accuracies"][task]
        print(f"{task} | {a:.1f}% | {b:.1f}% | {b-a:+.1f} points")
    comparison_path = Path("/content/report-comparison.json")
    comparison_path.write_text(json.dumps(comparison, indent=2) + "\n")
    store.upload(comparison_path, COMPARISON_KEY)
    print("Comparison checked in R2:", COMPARISON_KEY)
    if comparison["ready_for_beta_batch"]:
        print("Starting model and recorded training setup match. This does not explain the run-to-run difference.")
        print("CPU check finished. Next: fresh A100 session, MODE = BATCH.")
    else:
        print("CPU check finished. Resolve the printed setup differences before the beta batch.")
else:
    print("Report-comparison cell skipped in", MODE, "mode.")

## 7. Run the bounded beta batch in one A100 session

After the CPU comparison, open a fresh A100 session. Set MODE = "BATCH" in section 2 and run from the top. This cell runs the four new lessons automatically: beta 0, 0.001, 0.1, and 1. Each loads the original model before task 17. The dataset is downloaded once and reused. No earlier task is trained again.

The completed beta-0.01 result is read from R2, including its final model for the history check. It is not trained again. That reference came from an earlier session; the final table marks it as reused. The new trials share the current runtime. Software or GPU differences from the reference are printed, so this is an exploratory comparison rather than proof of a repeatable beta effect.

The cell saves epoch reports, final models, and one comparison table to R2. It checks every final model's history, then releases the GPU automatically. If training stops halfway, keep the partial files and use a new BATCH_EXPERIMENT_ID for a new attempt. There is no automatic midway restart.

Use task 17's score, each older-task score, and the largest older-task drop to decide whether stronger protection helps retention while allowing learning. An unchanged average can hide damage to one task. Stop after this batch and review the result before expanding the search.

In [ ]:
if MODE == "BATCH":
    from google.colab import runtime
    from uncle.config import Config
    from uncle.streams import build_tasks
    from uncle import checkpoint as checkpointing
    assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Choose a fresh A100 runtime."
    comparison_path = store.download(COMPARISON_KEY, CACHE)
    comparison = json.loads(comparison_path.read_text())
    assert comparison["ready_for_beta_batch"], "Resolve the CPU comparison's setup differences first."
    assert comparison["report_keys"]["Later R2 run"] == R2_REPORT_KEY
    baseline_path = store.download(R2_REPORT_KEY, CACHE)
    baseline = json.loads(baseline_path.read_text())
    assert file_hash(baseline_path) == comparison["report_hashes"]["Later R2 run"], "The reference report changed. Repeat COMPARE on CPU."
    assert baseline["status"] == "complete" and baseline["new_task"] == NEW_TASK
    assert baseline["config"]["beta"] == 0.01 and baseline["code_version"] == CODE_VERSION
    assert file_hash(SOURCE) == SOURCE_SHA256 == baseline["source_sha256"]
    assert baseline["starting_accuracies"] == details["accuracies"]
    assert baseline["protected_tasks"] == details["completed_tasks"]
    assert baseline["source_history"] == json.loads(json.dumps(details["history"]))
    source_config = Config(**details["config"])
    expected_config = vars(source_config).copy()
    expected_config.update(requests=tuple((r["action"], r["task"]) for r in details["history"]) + (("learn", NEW_TASK),), beta=0.01)
    expected_config = json.loads(json.dumps(expected_config))
    assert {k:v for k,v in baseline["config"].items() if k != "device"} == {k:v for k,v in expected_config.items() if k != "device"}, "Reference training settings differ from the source lesson."
    assert BATCH_BETAS == (0.0, 0.001, 0.01, 0.1, 1.0), "Keep the agreed five-value batch."
    assert BATCH_EXPERIMENT_ID and all(c.isalnum() or c in "_-" for c in BATCH_EXPERIMENT_ID)
    batch_root = OUTPUT_ROOT.parent / BATCH_EXPERIMENT_ID
    batch_prefix = "uncle/learning_loss_diagnostic/" + BATCH_EXPERIMENT_ID
    assert not list(store.keys(batch_prefix + "/")), "This batch already has files. Read them first; use a new batch name for a new attempt."
    assert not batch_root.exists() or not any(batch_root.iterdir()), "Local batch files already exist. Use a fresh session and a new batch name."
    runtime_fields = {"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)}
    for key, value in runtime_fields.items():
        if baseline["environment"].get(key) != value:
            print(f"Reference {key}: {baseline['environment'].get(key)}; current batch: {value}")
    print("Beta 0.01 is reused from an earlier session. Four new trials will run here.", flush=True)
    batch_root.mkdir(parents=True, exist_ok=True)
    batch_selection = {"source_key": SOURCE_KEY, "source_sha256": SOURCE_SHA256,
        "new_task": NEW_TASK, "betas": BATCH_BETAS, "reference_report_key": R2_REPORT_KEY,
        "reference_report_sha256": file_hash(baseline_path), "comparison_key": COMPARISON_KEY,
        "code_version": CODE_VERSION, "current_runtime": runtime_fields}
    (batch_root / "selection.json").write_text(json.dumps(batch_selection, indent=2) + "\n")
    store.upload(batch_root / "selection.json", batch_prefix + "/selection.json")
    task_order = (*details["completed_tasks"], NEW_TASK)
    tasks = build_tasks(source_config, root=DATA, include=task_order, download=True)
    assert baseline["environment"]["task_sizes"] == {t: {split: len(data) for split, data in tasks[t].items()} for t in task_order}, "Task sizes differ from the reference."
    batch_reports, summaries = [], []
    for beta in BATCH_BETAS:
        if beta == 0.01:
            prefix = R2_REPORT_KEY.rsplit("/", 1)[0]
            output = batch_root / "reused_beta_0_01"
            output.mkdir()
            cached_model = store.download(prefix + "/checkpoint.pt", CACHE)
            import shutil
            shutil.copyfile(cached_model, output / "checkpoint.pt")
            report = baseline
            (output / "report.json").write_text(json.dumps(report, indent=2) + "\n")
            print("Reused completed beta 0.01. No training for this value.", flush=True)
        else:
            label = format(beta, ".8g").replace(".", "_")
            output, prefix = batch_root / f"beta_{label}", batch_prefix + f"/beta_{label}"
            def save_batch_epoch(row):
                store.publish_lesson(output, prefix)
                print(f"Beta {beta:g}, pass {row['epoch']}: new-task loss = {row['task_loss']:.4f}; beta times protection = {row['weighted_protection_loss']:.4f}")
                print("Validation scores:", " | ".join(f"task {t}: {row['accuracies'][t]:.1f}%" for t in task_order), flush=True)
            print("Loading the original source model. Beta:", beta, flush=True)
            report = run_learning_diagnostic(SOURCE, output, NEW_TASK, tasks,
                beta=beta, device="cuda", code_version=CODE_VERSION, on_epoch=save_batch_epoch)
            store.publish_lesson(output, prefix, complete=True)
        assert report["source_sha256"] == SOURCE_SHA256
        batch_reports.append((output, prefix, report))
        row = summarize_diagnostic(report)
        row.update(reused=beta == 0.01, result_prefix=prefix, final_accuracies=report["final_accuracies"])
        summaries.append(row)
        torch.cuda.empty_cache()
    print("Beta | task 17 | mean old-score change | largest old-score drop | source of result")
    for row in summaries:
        print(f"{row['beta']:g} | {row['new_task_accuracy']:.1f}% | {row['mean_old_change']:+.2f} points | {row['largest_old_drop']:.1f} points | {'earlier reference' if row['reused'] else 'this batch'}")
        print("All final scores:", row["final_accuracies"])
    for output, prefix, report in batch_reports:
        assert report["status"] == "complete" and len(report["epochs"]) == report["config"]["epochs"]
        final_config = dict(report["config"])
        final_config["tasks"] = tuple(final_config["tasks"])
        final_config["hidden"] = tuple(final_config["hidden"])
        final_config["requests"] = tuple(tuple(r) for r in final_config["requests"])
        final = checkpointing.load(output / "checkpoint.pt", Config(**final_config))
        assert len(final["history"]) == len(details["history"]) + 1
        assert json.loads(json.dumps(final["history"][:-1])) == json.loads(json.dumps(details["history"]))
        assert final["history"][-1]["task"] == NEW_TASK and final["previous"] == report["final_accuracies"]
        assert not final["forgotten"]
        assert json.loads(store.download(prefix + "/report.json", CACHE).read_text()) == report
        assert store.info(prefix + "/checkpoint.pt")["Metadata"]["sha256"] == file_hash(output / "checkpoint.pt")
        del final
    assert file_hash(SOURCE) == SOURCE_SHA256
    result = {"selection": batch_selection, "trials": summaries,
        "limitation": "Beta 0.01 is an earlier-session reference; four new trials share the current runtime."}
    (batch_root / "beta_batch_comparison.json").write_text(json.dumps(result, indent=2) + "\n")
    store.upload(batch_root / "beta_batch_comparison.json", batch_prefix + "/beta_batch_comparison.json")
    print("Five results checked. Comparison saved in R2:", batch_prefix + "/beta_batch_comparison.json", flush=True)
    print("Batch finished. Ending the GPU session.", flush=True)
    runtime.unassign()
else:
    print("Beta-batch cell skipped in", MODE, "mode. No new training.")

## 8. Repeat only beta 0.1 and compare the two runs

Use a fresh A100 session. Set MODE = "REPEAT" in section 2 and run all cells from the top. Earlier training and comparison cells will skip automatically. Leave the original source model unchanged. This repeats task 17 at beta 0.1 with the same recorded seed and training settings. It does not train tasks 3, 0, 9, or 5 again, and it does not start from the first trial's final model.

The cell first checks the saved beta-0.1 report against the source model and current settings. It saves this new lesson under REPEAT_EXPERIMENT_ID, so the first result is preserved. After training, it prints both runs' scores, average old-task changes, and largest old-task drops. The comparison and model are checked in R2 before the GPU is released automatically.

Read the comparison before choosing the next experiment. If the older-task scores stay close to their starting values and task 17 still learns, beta 0.1 remains a candidate for a longer learning-only sequence. Two runs from this one source model do not establish results across different seeds or task sequences. If the session stops, keep its files and use a new REPEAT_EXPERIMENT_ID for another attempt.


In [ ]:
if MODE == "REPEAT":
    from google.colab import runtime
    from uncle.config import Config
    from uncle.streams import build_tasks
    from uncle import checkpoint as checkpointing
    assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Choose a fresh A100 runtime."
    assert REPEAT_EXPERIMENT_ID and all(c.isalnum() or c in "_-" for c in REPEAT_EXPERIMENT_ID)
    repeat_root = OUTPUT_ROOT.parent / REPEAT_EXPERIMENT_ID
    repeat_prefix = "uncle/learning_loss_diagnostic/" + REPEAT_EXPERIMENT_ID
    assert not list(store.keys(repeat_prefix + "/")), "This repeat already has files. Read them first; use a new repeat name for another attempt."
    assert not repeat_root.exists() or not any(repeat_root.iterdir()), "Local repeat files exist. Use a fresh session and a new repeat name."
    reference_path = store.download(REPEAT_REPORT_KEY, CACHE)
    reference = json.loads(reference_path.read_text())
    assert reference["status"] == "complete" and reference["new_task"] == NEW_TASK
    assert reference["config"]["beta"] == 0.1 and reference["code_version"] == CODE_VERSION
    assert file_hash(SOURCE) == SOURCE_SHA256 == reference["source_sha256"]
    assert reference["starting_accuracies"] == details["accuracies"]
    assert reference["protected_tasks"] == details["completed_tasks"]
    assert reference["source_history"] == json.loads(json.dumps(details["history"]))
    source_config = Config(**details["config"])
    expected = vars(source_config).copy()
    expected.update(requests=tuple((r["action"], r["task"]) for r in details["history"]) + (("learn", NEW_TASK),), beta=0.1)
    expected = json.loads(json.dumps(expected))
    assert {k:v for k,v in reference["config"].items() if k != "device"} == {k:v for k,v in expected.items() if k != "device"}, "Training settings differ from the first beta-0.1 trial."
    task_order = (*details["completed_tasks"], NEW_TASK)
    tasks = build_tasks(source_config, root=DATA, include=task_order, download=True)
    assert reference["environment"]["task_sizes"] == {t: {split: len(data) for split, data in tasks[t].items()} for t in task_order}, "Task sizes differ from the first trial."
    repeat_root.mkdir(parents=True, exist_ok=True)
    selection = {"source_key": SOURCE_KEY, "source_sha256": SOURCE_SHA256, "beta": 0.1,
        "reference_report_key": REPEAT_REPORT_KEY, "reference_report_sha256": file_hash(reference_path), "code_version": CODE_VERSION}
    (repeat_root / "selection.json").write_text(json.dumps(selection, indent=2) + "\n")
    store.upload(repeat_root / "selection.json", repeat_prefix + "/selection.json")
    output, prefix = repeat_root / "beta_0_1", repeat_prefix + "/beta_0_1"
    def save_repeat_epoch(row):
        store.publish_lesson(output, prefix)
        print(f"Pass {row['epoch']}: new-task loss = {row['task_loss']:.4f}; beta times protection = {row['weighted_protection_loss']:.4f}")
        print("Validation scores:", " | ".join(f"task {t}: {row['accuracies'][t]:.1f}%" for t in task_order), flush=True)
    print("Repeating beta 0.1 from the original source model.", flush=True)
    repeated = run_learning_diagnostic(SOURCE, output, NEW_TASK, tasks,
        beta=0.1, device="cuda", code_version=CODE_VERSION, on_epoch=save_repeat_epoch)
    assert repeated["source_sha256"] == SOURCE_SHA256
    store.publish_lesson(output, prefix, complete=True)
    final_config = dict(repeated["config"])
    final_config["tasks"] = tuple(final_config["tasks"])
    final_config["hidden"] = tuple(final_config["hidden"])
    final_config["requests"] = tuple(tuple(r) for r in final_config["requests"])
    final = checkpointing.load(output / "checkpoint.pt", Config(**final_config))
    assert len(final["history"]) == len(details["history"]) + 1
    assert json.loads(json.dumps(final["history"][:-1])) == reference["source_history"]
    assert final["history"][-1]["task"] == NEW_TASK and final["previous"] == repeated["final_accuracies"]
    assert not final["forgotten"]
    assert len(repeated["epochs"]) == repeated["config"]["epochs"]
    assert json.loads(store.download(prefix + "/report.json", CACHE).read_text()) == repeated
    assert store.info(prefix + "/checkpoint.pt")["Metadata"]["sha256"] == file_hash(output / "checkpoint.pt")
    assert file_hash(SOURCE) == SOURCE_SHA256
    differences = {t: repeated["final_accuracies"][t] - reference["final_accuracies"][t] for t in task_order}
    environment_changes = {k: {"first": reference["environment"].get(k), "repeat": repeated["environment"].get(k)}
        for k in ("python", "torch", "cuda", "gpu", "git_commit") if reference["environment"].get(k) != repeated["environment"].get(k)}
    repeat_comparison = {"selection": selection, "first": summarize_diagnostic(reference),
        "repeat": summarize_diagnostic(repeated), "first_final_accuracies": reference["final_accuracies"],
        "repeat_final_accuracies": repeated["final_accuracies"], "repeat_minus_first": differences,
        "environment_differences": environment_changes,
        "limitation": "Two sessions from one saved source model; no different-seed or longer-sequence result."}
    print("Task | starting score | first beta 0.1 | repeated beta 0.1 | repeat minus first")
    for t in task_order:
        start = f"{details['accuracies'][t]:.1f}%" if t in details["accuracies"] else "new task"
        print(f"{t} | {start} | {reference['final_accuracies'][t]:.1f}% | {repeated['final_accuracies'][t]:.1f}% | {differences[t]:+.1f} points")
    for label in ("first", "repeat"):
        row = repeat_comparison[label]
        print(f"{label}: task 17 = {row['new_task_accuracy']:.1f}%; mean old-score change = {row['mean_old_change']:+.2f} points; largest old-score drop = {row['largest_old_drop']:.1f} points")
    print("Computer and software differences:", json.dumps(environment_changes, indent=2))
    comparison_file = repeat_root / "repeat_comparison.json"
    comparison_file.write_text(json.dumps(repeat_comparison, indent=2) + "\n")
    store.upload(comparison_file, repeat_prefix + "/repeat_comparison.json")
    print("Repeat checked. Comparison saved in R2:", repeat_prefix + "/repeat_comparison.json", flush=True)
    print("Review both results before starting a longer sequence. Ending the GPU session.", flush=True)
    del final
    runtime.unassign()
else:
    print("Beta-0.1 repeat skipped in", MODE, "mode. No new training.")
